# Notebook 04: Model Serving Architectures

## Why This Matters

Model serving is where ML systems meet production reliability engineering. Training an accurate model is only half the battle—serving it at scale with sub-100ms latency, 99.99% availability, and cost efficiency is the other half. Companies like Meta serve billions of inference requests daily across ads, feed ranking, and integrity systems. Uber's Hexagon, Twitter's Cortex, and DoorDash's ML platform all needed custom serving infrastructure to meet their SLAs. Understanding the tradeoffs between REST vs gRPC, model server frameworks (TorchServe, TFServing, Triton), and deployment patterns like shadow deployment and canary releases is expected knowledge for senior ML roles.

## 1. Model Serving Architecture Patterns

There are four primary serving architectures, each with different latency, throughput, and operational characteristics. The right choice depends on your latency budget, traffic volume, and model complexity.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Model Serving Architecture Patterns', fontsize=15, fontweight='bold', y=0.98)

def box(ax, x, y, w, h, text, color, fontsize=9, text_color='white'):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color, multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2, label, fontsize=7.5, color=color)

# Pattern 1: REST API serving
ax = axes[0][0]
ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis('off')
ax.set_title('1. REST API Model Server', fontsize=11, fontweight='bold')
box(ax, 0.5, 6, 2.5, 1.2, 'Client\nApplication', '#7F8C8D')
box(ax, 4, 6, 2.5, 1.2, 'Load\nBalancer', '#E67E22')
box(ax, 2, 3.5, 2, 1.2, 'Model\nServer 1', '#2980B9')
box(ax, 4.5, 3.5, 2, 1.2, 'Model\nServer 2', '#2980B9')
box(ax, 7, 3.5, 2, 1.2, 'Model\nServer 3', '#2980B9')
box(ax, 3.5, 1, 3, 1, 'Model Store\n(S3/NFS)', '#27AE60')
arr(ax, 3, 6.6, 4, 6.6)
arr(ax, 5.25, 6, 3, 4.7)
arr(ax, 5.25, 6, 5.5, 4.7)
arr(ax, 5.25, 6, 8, 4.7)
arr(ax, 3, 3.5, 4.2, 2)
arr(ax, 5.5, 3.5, 5.5, 2)
arr(ax, 8, 3.5, 6.8, 2)
pros = ['✓ Simple to implement', '✓ Language agnostic', '✓ Easy load balancing', '✗ Higher latency than gRPC', '✗ No streaming support']
for i, p in enumerate(pros):
    ax.text(0.3, 0.8 - i*0.15, p, fontsize=7.5, color='#27AE60' if '✓' in p else '#E74C3C')

# Pattern 2: gRPC serving  
ax = axes[0][1]
ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis('off')
ax.set_title('2. gRPC Model Server (TF Serving / Triton)', fontsize=11, fontweight='bold')
box(ax, 0.5, 6, 2.5, 1.2, 'gRPC\nClient', '#7F8C8D')
box(ax, 4, 6, 2.5, 1.2, 'Envoy Proxy\n(gRPC LB)', '#E67E22')
box(ax, 2, 3.5, 2.5, 1.5, 'Triton\nInference\nServer', '#6C3483')
box(ax, 5.5, 3.5, 2.5, 1.5, 'TF/Torch/ONNX\nBackend\n(GPU)', '#8E44AD')
box(ax, 3.5, 1, 3, 1, 'Model\nRepository', '#1A5276')
arr(ax, 3, 6.6, 4, 6.6)
arr(ax, 5.25, 6, 3.25, 5)
arr(ax, 3.25, 3.5, 3.5, 2)
arr(ax, 5.5, 4.25, 5.5, 4.25)
ax.annotate('Proto buffers\n(2-5x faster)', xy=(3.8, 6.6), xytext=(3.8, 5.3),
            fontsize=7.5, color='#8E44AD', ha='center')
pros2 = ['✓ 2-5x lower latency vs REST', '✓ Streaming + bidirectional', '✓ Strong typing (Proto)', '✓ HTTP/2 multiplexing', '✗ More complex setup']
for i, p in enumerate(pros2):
    ax.text(0.3, 0.8 - i*0.15, p, fontsize=7.5, color='#27AE60' if '✓' in p else '#E74C3C')

# Pattern 3: Embedded model
ax = axes[1][0]
ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis('off')
ax.set_title('3. Embedded / In-Process Model', fontsize=11, fontweight='bold')
box(ax, 2, 5.5, 6, 1.5, 'Application Service\n(Python/Java/Go)', '#2C3E50')
box(ax, 3, 3.5, 4, 1.5, 'Model\n(loaded in-process)\nTensorFlow Lite / ONNX', '#16A085')
box(ax, 3, 1, 4, 1.2, 'Local model file\n(downloaded at startup)', '#1A5276')
arr(ax, 5, 5.5, 5, 5)
arr(ax, 5, 3.5, 5, 2.2)
pros3 = ['✓ Lowest latency (no network hop)', '✓ No additional infra', '✓ Good for mobile/edge', '✗ Model updates require redeploy', '✗ Limited GPU access', '✗ Model size constraints']
for i, p in enumerate(pros3):
    ax.text(0.3, 0.9 - i*0.14, p, fontsize=7.5, color='#27AE60' if '✓' in p else '#E74C3C')

# Pattern 4: Multi-model pipeline
ax = axes[1][1]
ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis('off')
ax.set_title('4. Multi-Model Pipeline (Seldon / BentoML)', fontsize=11, fontweight='bold')
box(ax, 0.3, 6.5, 2, 1, 'Request', '#7F8C8D')
box(ax, 3, 6.5, 2.5, 1, 'Pre-processor', '#E67E22')
box(ax, 1.5, 4.5, 2.5, 1, 'Embedding\nModel A', '#2980B9')
box(ax, 4.5, 4.5, 2.5, 1, 'Classifier\nModel B', '#8E44AD')
box(ax, 7, 4.5, 2.5, 1, 'Post-processor', '#27AE60')
box(ax, 3.5, 2.5, 3, 1, 'Response\n(aggregated)', '#2C3E50')
arr(ax, 2.3, 7, 3, 7)
arr(ax, 4.25, 6.5, 2.75, 5.5)
arr(ax, 4.25, 6.5, 5.75, 5.5)
arr(ax, 2.75, 4.5, 3.7, 3.5)
arr(ax, 5.75, 4.5, 5.3, 3.5)
arr(ax, 7, 5, 7, 5)
arr(ax, 8.25, 4.5, 7.5, 3.5)
pros4 = ['✓ Composable models', '✓ Independent scaling', '✓ A/B test individual steps', '✗ Higher latency (DAG overhead)', '✗ Complex failure modes']
for i, p in enumerate(pros4):
    ax.text(0.3, 0.8 - i*0.15, p, fontsize=7.5, color='#27AE60' if '✓' in p else '#E74C3C')

plt.tight_layout()
plt.savefig('serving_architectures.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Latency Budget Decomposition

Meeting a 100ms latency SLA requires understanding where time is spent. In a typical recommendation system, the model compute is often NOT the bottleneck—feature retrieval and network overhead are. This insight changes infrastructure decisions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Latency breakdown for a recommendation system
components = [
    ('Network (client → LB)', 5, '#95A5A6'),
    ('Load balancer routing', 1, '#BDC3C7'),
    ('Feature lookup (online store)', 18, '#E74C3C'),
    ('Candidate retrieval (ANN)', 12, '#E67E22'),
    ('Feature assembly', 8, '#F39C12'),
    ('Model inference (ranking)', 22, '#8E44AD'),
    ('Post-processing / re-ranking', 6, '#2980B9'),
    ('Response serialization', 4, '#27AE60'),
    ('Network (LB → client)', 5, '#95A5A6'),
    ('Buffer / tail latency', 19, '#E74C3C'),
]

labels = [c[0] for c in components]
values = [c[1] for c in components]
colors = [c[2] for c in components]
total = sum(values)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Horizontal bar chart
ax = axes[0]
bars = ax.barh(range(len(labels)), values, color=colors, alpha=0.85, height=0.65)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Latency (ms)', fontsize=11)
ax.set_title(f'Latency Budget Breakdown\n(Total p50: {total}ms, budget: 100ms)', fontsize=11, fontweight='bold')
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='20ms per-component target')

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}ms', va='center', fontsize=9)

ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(0, 30)

# Pie chart of where time is spent
ax2 = axes[1]
wedge_sizes = values
wedge_labels = [f"{l.split('(')[0].strip()}\n{v}ms" for l, v in zip(labels, values)]
explode = [0.05 if v == max(values) else 0 for v in values]
ax2.pie(wedge_sizes, labels=wedge_labels, colors=colors, autopct='%1.0f%%',
        startangle=90, explode=explode, textprops={'fontsize': 7.5})
ax2.set_title(f'Proportion of p50 Latency ({total}ms total)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('latency_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()

print("Optimization priorities (from biggest impact):")
sorted_components = sorted(components, key=lambda x: -x[1])
for i, (comp, ms, _) in enumerate(sorted_components[:5], 1):
    pct = ms/total*100
    print(f"  {i}. {comp}: {ms}ms ({pct:.0f}%) — {ms/total*100:.0f}% of budget")

print("\nKey insight: Feature lookup + model inference = 40ms of a 100ms budget.")
print("Caching features, batching requests, and model quantization are top levers.")

## 3. Batching and Throughput Optimization

GPU utilization is maximized through batching—processing multiple requests together. Dynamic batching (wait up to N ms or M requests) is a core technique in Triton Inference Server and TorchServe. The tradeoff is latency (individual requests wait) vs throughput (more requests per second per GPU).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model latency vs batch size (GPU typical behavior)
batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])

# Throughput: increases steeply then plateaus (GPU saturation)
# items/sec = batch_size / latency
base_latency_ms = 5  # baseline GPU latency
gpu_saturation_batch = 32

latency_ms = base_latency_ms + 0.8 * batch_sizes + 0.01 * batch_sizes**1.5
throughput = (batch_sizes / latency_ms) * 1000  # requests per second

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Latency vs batch size
ax = axes[0]
ax.plot(batch_sizes, latency_ms, 'b-o', linewidth=2, markersize=6)
ax.axhline(y=100, color='red', linestyle='--', label='100ms SLA')
ax.axhline(y=50, color='orange', linestyle='--', label='50ms target')
ax.set_xscale('log', base=2)
ax.set_xlabel('Batch Size', fontsize=11)
ax.set_ylabel('Latency (ms)', fontsize=11)
ax.set_title('Latency vs Batch Size', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes, fontsize=8)

# Throughput vs batch size
ax2 = axes[1]
ax2.plot(batch_sizes, throughput, 'g-o', linewidth=2, markersize=6)
ax2.axvline(x=gpu_saturation_batch, color='purple', linestyle='--', alpha=0.7, label=f'GPU saturation ~bs={gpu_saturation_batch}')
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Batch Size', fontsize=11)
ax2.set_ylabel('Throughput (req/sec)', fontsize=11)
ax2.set_title('Throughput vs Batch Size', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(batch_sizes)
ax2.set_xticklabels(batch_sizes, fontsize=8)

# Efficiency = throughput per unit latency
ax3 = axes[2]
efficiency = throughput / latency_ms
ax3.plot(batch_sizes, efficiency, 'r-o', linewidth=2, markersize=6)
optimal_idx = np.argmax(efficiency)
ax3.axvline(x=batch_sizes[optimal_idx], color='purple', linestyle='--',
            label=f'Optimal batch size: {batch_sizes[optimal_idx]}')
ax3.set_xscale('log', base=2)
ax3.set_xlabel('Batch Size', fontsize=11)
ax3.set_ylabel('Efficiency (throughput/latency)', fontsize=11)
ax3.set_title('Efficiency vs Batch Size\n(Throughput/Latency)', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_xticks(batch_sizes)
ax3.set_xticklabels(batch_sizes, fontsize=8)

plt.suptitle('GPU Batching Tradeoffs', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('batching_tradeoffs.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"At batch_size=1:   latency={latency_ms[0]:.1f}ms, throughput={throughput[0]:.0f} req/s")
print(f"At batch_size=32:  latency={latency_ms[5]:.1f}ms, throughput={throughput[5]:.0f} req/s")
print(f"At batch_size=256: latency={latency_ms[-1]:.1f}ms, throughput={throughput[-1]:.0f} req/s")
print(f"\nOptimal efficiency batch size: {batch_sizes[optimal_idx]}")
print("\nDynamic batching: wait max 2ms OR 32 requests, whichever comes first.")
print("This achieves high throughput without violating p99 latency SLAs.")

## 4. Deployment Patterns: Safe Model Rollouts

Deploying a new model version without downtime or performance regression requires staged rollout patterns. Every major ML platform—Uber, Airbnb, Netflix—uses variants of these patterns.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 3, figsize=(16, 8))
fig.suptitle('Safe Model Deployment Patterns', fontsize=14, fontweight='bold')

def draw_box(ax, x, y, w, h, text, color):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', multialignment='center')

# Pattern 1: Shadow Deployment
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('Shadow Deployment', fontsize=11, fontweight='bold')
draw_box(ax, 2, 7.5, 6, 1, 'Traffic (100%)', '#7F8C8D')
draw_box(ax, 0.5, 5, 4, 1.5, 'Production Model\n(serving users)', '#27AE60')
draw_box(ax, 5.5, 5, 4, 1.5, 'Shadow Model\n(predictions logged,\nnot served)', '#2980B9')
draw_box(ax, 3, 2.5, 4, 1.5, 'Log Comparison\n(offline analysis)', '#8E44AD')

ax.annotate('', xy=(2.5, 6.5), xytext=(5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2))
ax.annotate('', xy=(7.5, 6.5), xytext=(5, 7.5),
            arrowprops=dict(arrowstyle='-->', color='#2980B9', lw=2, linestyle='dashed'))
ax.annotate('', xy=(5, 4), xytext=(2.5, 5),
            arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=1.5))
ax.annotate('', xy=(5, 4), xytext=(7.5, 5),
            arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=1.5))

ax.text(5, 1.5, 'Use when: High-stakes model, want to\nvalidate without any user impact.',
        ha='center', fontsize=8.5, color='#555', style='italic')

# Pattern 2: Canary / Blue-Green
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('Canary Deployment', fontsize=11, fontweight='bold')
draw_box(ax, 2, 7.5, 6, 1, 'Traffic Router / Load Balancer', '#7F8C8D')
draw_box(ax, 0.5, 5, 3.5, 1.5, 'Model v1\n(95% traffic)', '#27AE60')
draw_box(ax, 5.5, 5, 3.5, 1.5, 'Model v2\n(5% canary)', '#E67E22')
draw_box(ax, 3, 2.5, 4, 1.5, 'Metric Dashboard\n(compare both cohorts)', '#8E44AD')

ax.annotate('', xy=(2.25, 6.5), xytext=(4.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2))
ax.annotate('', xy=(7.25, 6.5), xytext=(5.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=2))
ax.text(4.6, 7.8, '95%', ha='center', fontsize=8, color='#27AE60', fontweight='bold')
ax.text(6.2, 7.8, '5%', ha='center', fontsize=8, color='#E67E22', fontweight='bold')

ax.text(5, 1.5, 'Use when: Need live traffic validation\nbefore full rollout. Most common pattern.',
        ha='center', fontsize=8.5, color='#555', style='italic')

# Pattern 3: Gradual Rollout with auto-rollback
ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('Gradual Rollout with Auto-Rollback', fontsize=11, fontweight='bold')

rollout_stages = [
    (0.5, 7.2, '1%\nCanary', '#FFF3CD', '#E67E22'),
    (2.5, 7.2, '5%', '#D4EFDF', '#27AE60'),
    (4.5, 7.2, '25%', '#D4EFDF', '#27AE60'),
    (6.5, 7.2, '50%', '#D4EFDF', '#27AE60'),
    (8.5, 7.2, '100%', '#D4EFDF', '#2ECC71'),
]
for x, y, label, fc, ec in rollout_stages:
    r = FancyBboxPatch((x, y), 1.7, 1.2, boxstyle="round,pad=0.1",
                       facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(r)
    ax.text(x+0.85, y+0.6, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='#333')
    if x < 8.5:
        ax.annotate('', xy=(x+1.9, y+0.6), xytext=(x+1.7, y+0.6),
                    arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

draw_box(ax, 1, 4.5, 7.5, 1.5, 'Automated Health Checks (after each stage):\nLatency p99 < SLA | Error rate < 0.1% | Online metric not regressing', '#2C3E50')
draw_box(ax, 2, 2.5, 5.5, 1.5, 'Auto-Rollback\n(if any check fails: immediate revert to v1)', '#E74C3C')

ax.annotate('', xy=(4.5, 4.5), xytext=(4.5, 7.2),
            arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))
ax.annotate('', xy=(4.75, 4), xytext=(4.75, 4.5),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5))

ax.text(5, 1.5, 'Use when: Automated deployment, CI/CD for ML models.',
        ha='center', fontsize=8.5, color='#555', style='italic')

plt.tight_layout()
plt.savefig('deployment_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Model Serving Infrastructure Comparison

Choosing a model server framework affects what models you can serve, what hardware you can use, and how much operational complexity you take on.

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        'Framework': 'TensorFlow Serving',
        'Backends': 'TensorFlow only',
        'Protocol': 'gRPC + REST',
        'Batching': 'Dynamic',
        'GPU Support': 'Yes',
        'Latency': 'Very low',
        'Ops Complexity': 'Medium',
        'Best For': 'TF-first orgs, production-proven',
    },
    {
        'Framework': 'TorchServe',
        'Backends': 'PyTorch',
        'Protocol': 'REST',
        'Batching': 'Dynamic',
        'GPU Support': 'Yes',
        'Latency': 'Low',
        'Ops Complexity': 'Medium',
        'Best For': 'PyTorch-first orgs (most research teams)',
    },
    {
        'Framework': 'Triton Inference Server',
        'Backends': 'TF, PyTorch, ONNX, TensorRT',
        'Protocol': 'gRPC + REST (KServe API)',
        'Batching': 'Dynamic + concurrent',
        'GPU Support': 'Excellent (NVIDIA)',
        'Latency': 'Lowest',
        'Ops Complexity': 'High',
        'Best For': 'Maximum GPU utilization, multi-model serving',
    },
    {
        'Framework': 'BentoML',
        'Backends': 'Any (framework agnostic)',
        'Protocol': 'REST',
        'Batching': 'Adaptive',
        'GPU Support': 'Yes',
        'Latency': 'Low-Medium',
        'Ops Complexity': 'Low',
        'Best For': 'Fast iteration, multiple frameworks, startups',
    },
    {
        'Framework': 'Ray Serve',
        'Backends': 'Any Python',
        'Protocol': 'REST + Python native',
        'Batching': 'Dynamic',
        'GPU Support': 'Yes',
        'Latency': 'Low',
        'Ops Complexity': 'Medium',
        'Best For': 'Composable pipelines, existing Ray cluster',
    },
    {
        'Framework': 'FastAPI + Custom',
        'Backends': 'Any Python',
        'Protocol': 'REST',
        'Batching': 'Manual',
        'GPU Support': 'Manual',
        'Latency': 'Variable',
        'Ops Complexity': 'Low (initially)',
        'Best For': 'Prototypes, simple models, full control needed',
    },
])

print(comparison.to_string(index=False))

print("\nInterview decision tree:")
print("  Maximum GPU utilization & multi-model?  → Triton")
print("  PyTorch production, simple setup?       → TorchServe")
print("  TensorFlow, proven at scale?            → TF Serving")
print("  Polyglot ML team, fast shipping?        → BentoML or Ray Serve")
print("  Quick prototype, single model?          → FastAPI + model.predict()")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| REST vs gRPC | gRPC is 2-5x lower latency; REST is simpler to debug | Know when latency justifies the complexity |
| Latency budget | Feature lookup + model inference dominate; network is fixed | Can decompose a latency SLA into components |
| Dynamic batching | Wait N ms or M requests for GPU efficiency | Knows the latency vs throughput tradeoff |
| Shadow deployment | Run new model in parallel, no user impact | For highest-stakes rollouts |
| Canary deployment | 5% → 25% → 100% with health checks | Most common production pattern |
| Auto-rollback | Automated metric checks trigger revert | Shows production reliability thinking |
| Triton vs TF Serving vs TorchServe | Triton: multi-backend GPU; TF/TorchServe: framework-specific | Justify choice for context |
| Model versioning | Load balancer routes by version tag | Required for A/B testing and rollbacks |